# LightRAG Test Notebook

This notebook tests the reusable LightRAG module in `modules/lightrag`.

## What it does
- Loads `LightRAGModule`
- Ingests documents from `data/user/ruling`
- Runs a sample query against the indexed data

> If needed, install dependencies first:
> `uv add lightrag-hku pypdf`

In [ ]:
from functools import partial
from pathlib import Path
import importlib
import sys

import numpy as np

sys.path.append("/Users/lscm/Project/PCS/HSCodeSearch")

from lightrag.llm.ollama import ollama_embed, ollama_model_complete
from lightrag.utils import EmbeddingFunc
import modules.lightrag.rag_module as rag_module

# Force reload so notebook uses latest code edits.
importlib.reload(rag_module)
LightRAGBackend = rag_module.LightRAGBackend
LightRAGModule = rag_module.LightRAGModule

# Keep index files in a reusable local folder.
WORKING_DIR = Path("../../data/user/lightrag-index")
SOURCE_DIR = Path("../../data/user/ruling")
OLLAMA_HOST = "http://10.103.1.23:11434"
EMBED_MODEL = "qwen3-embedding:4b"
LLM_MODEL = "qwen3:32b-q4_K_M"

assert SOURCE_DIR.exists(), f"Missing source directory: {SOURCE_DIR}"


async def robust_ollama_embed(texts, max_token_size=None, **kwargs):
    """Return one embedding per input text, with fallback when Ollama batches mismatch."""
    base = await ollama_embed.func(
        texts,
        host=OLLAMA_HOST,
        embed_model=EMBED_MODEL,
        max_token_size=max_token_size,
        **kwargs,
    )
    arr = np.asarray(base)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    if arr.shape[0] == len(texts):
        return arr

    vectors = []
    for text in texts:
        single = await ollama_embed.func(
            [text],
            host=OLLAMA_HOST,
            embed_model=EMBED_MODEL,
            max_token_size=max_token_size,
            **kwargs,
        )
        single_arr = np.asarray(single)
        if single_arr.ndim == 1:
            vectors.append(single_arr)
        else:
            vectors.append(np.asarray(single_arr[0]))
    return np.vstack(vectors)


probe = rag_module._run_maybe_awaitable(robust_ollama_embed(["dimension probe"]))
embedding_dim = int(probe.shape[1])
print(f"Using embedding dimension: {embedding_dim}")

embedding_func = EmbeddingFunc(
    embedding_dim=embedding_dim,
    max_token_size=getattr(ollama_embed, "max_token_size", 8192),
    func=robust_ollama_embed,
    model_name=EMBED_MODEL,
)
llm_model_func = partial(ollama_model_complete, host=OLLAMA_HOST)

backend = LightRAGBackend(
    working_dir=WORKING_DIR,
    embedding_func=embedding_func,
    llm_model_func=llm_model_func,
    llm_model_name=LLM_MODEL,
)
rag = LightRAGModule(working_dir=WORKING_DIR, backend=backend)

report = rag.ingest_directory(
    SOURCE_DIR,
    recursive=True,
    # Use a smaller number first for a quick smoke test; remove limit for full indexing.
    max_files=20,
)

report

INFO: Creating working directory ../../data/user/lightrag-index
INFO: [] Created new empty graph file: ../../data/user/lightrag-index/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 2560, 'metric': 'cosine', 'storage_file': '../../data/user/lightrag-index/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 2560, 'metric': 'cosine', 'storage_file': '../../data/user/lightrag-index/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 2560, 'metric': 'cosine', 'storage_file': '../../data/user/lightrag-index/vdb_chunks.json'} 0 data


Using embedding dimension: 2560


INFO: Created 20 duplicate document records with track_id: insert_20260424_170020_a883c6ef
INFO: Preserving 20 failed document entries for manual review
INFO: Reset 20 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 20 document(s)
ERROR: == Lock == Process 89023: Failed to acquire lock 'full_docs:default_key': <asyncio.locks.Lock object at 0x110330b90 [locked]> is bound to a different event loop
ERROR: Lock acquisition failed for key default_key: <asyncio.locks.Lock object at 0x110330b90 [locked]> is bound to a different event loop
INFO: Extracting stage 1/20: ../../data/user/ruling/us/20260421-1552/H347082.pdf
INFO: Processing d-id: ../../data/user/ruling/us/20260421-1552/H347082.pdf
ERROR: Traceback (most recent call last):
  File "/Users/lscm/Project/PCS/HSCodeSearch/.venv/lib/python3.12/site-packages/lightrag/lightrag.py", line 1908, in process_document
    content_data = await self.full_docs.get_by_id(doc_id)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
question = "What kinds of product classification rulings are in this dataset?"
answer = rag.query(question, mode="hybrid")
answer

In [ ]:
# Optional: run this when you are done to close storage resources.
rag.close()